In [51]:
import pandas as pd
import sqlite3

In [52]:
conn = sqlite3.connect("../data/checking-logs.sqlite")
cur = conn.cursor()
cur.execute("drop table if exists datamart")
cur.execute("""
            create table datamart as
            select c.uid, c.labname, min(c.timestamp) as first_commit_ts, pv.first_view_ts
            from checker c
            left join (
                select uid, min(datetime) as first_view_ts
                from pageviews
                group by uid
            ) pv
            on c.uid = pv.uid
            where
                c.status = 'ready'
                and c.numTrials = 1
                and c.labname IN (
                    'laba04',
                    'laba04s',
                    'laba05',
                    'laba06',
                    'laba06s',
                    'project1'
                )
                and c.uid like 'user_%'
            group by c.uid, c.labname
            """)
conn.commit()
datamart = pd.read_sql("select * from datamart", conn)

datamart["first_commit_ts"] = pd.to_datetime(datamart["first_commit_ts"])
datamart["first_view_ts"] = pd.to_datetime(datamart["first_view_ts"])
datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 4.5+ KB


In [53]:
test = datamart[datamart["first_view_ts"].notna()].copy()
control = datamart[datamart["first_view_ts"].isna()].copy()

control["first_view_ts"] = test["first_view_ts"].mean()

test.to_sql("test", conn, if_exists="replace", index=False)
control.to_sql("control", conn, if_exists="replace", index=False)

control.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.2+ KB


In [54]:
conn.close()